# SEC Disclosure-Risk Monitor — Risk Backtest

This notebook provides a repeatable risk-monitoring validation workflow for current signals:
- NT filings
- Friday after-hours (Friday burying)
- 8-K monthly spike


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(Path.cwd().parent / '.env', override=False)
DB_BACKEND = os.getenv('DB_BACKEND', 'sqlite').strip().lower()

if DB_BACKEND == 'postgres':
    DATABASE_URL = os.getenv('DATABASE_URL') or os.getenv('DATABASE_URL_RW')
    if not DATABASE_URL:
        raise ValueError('DATABASE_URL (or DATABASE_URL_RW) is required when DB_BACKEND=postgres')
    sqlalchemy_url = DATABASE_URL
    if sqlalchemy_url.startswith('postgresql://'):
        sqlalchemy_url = 'postgresql+psycopg://' + sqlalchemy_url[len('postgresql://'):]
    engine = create_engine(sqlalchemy_url, pool_pre_ping=True)
    db_target = 'postgres'
else:
    db_path = Path.cwd().parent / 'data' / 'sec_anomaly.db'
    engine = create_engine(f'sqlite:///{db_path}')
    db_target = str(db_path)

def read_sql_df(sql: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params or {})

print(f'Using backend: {DB_BACKEND} ({db_target})')

def load_alerts(anomaly_type: str) -> pd.DataFrame:
    df = read_sql_df(
        'SELECT * FROM alerts WHERE anomaly_type = :anomaly_type ORDER BY created_at DESC',
        {'anomaly_type': anomaly_type},
    )
    if df.empty:
        return df
    df['details'] = df['details'].apply(
        lambda value: value if isinstance(value, (dict, list)) else json.loads(value)
    )
    details_df = pd.json_normalize(df['details'])
    return pd.concat([df.drop(columns=['details']), details_df], axis=1)


In [ ]:
# Overall alert counts
summary = read_sql_df('SELECT anomaly_type, COUNT(*) AS count FROM alerts GROUP BY anomaly_type')
print(summary.to_string(index=False))


## NT Filings Signal Review

Sanity checks:
- Alerts should only contain `NT %` or `NT-%` filings.
- Spot-check a few alerts to confirm filing type.


In [ ]:
nt_alerts = load_alerts('NT_FILING')
if nt_alerts.empty:
    print('No NT alerts yet. Run src/detection/nt_detection.py first.')
else:
    cols = [
        'company_ticker',
        'company_name',
        'filing_type',
        'filed_at',
        'severity_score',
        'created_at',
    ]
    available = [c for c in cols if c in nt_alerts.columns]
    print(nt_alerts[available].head(15).to_string(index=False))


## Friday After-Hours Signal Review

Sanity checks:
- Filing time should be Friday after 4pm US/Eastern.
- Forms should be 8-K or 8-K/A (MVP scope).


In [ ]:
friday_alerts = load_alerts('FRIDAY_BURYING')
if friday_alerts.empty:
    print('No Friday burying alerts yet. Run src/detection/friday_detection.py first.')
else:
    cols = [
        'company_ticker',
        'company_name',
        'filing_type',
        'filed_at',
        'severity_score',
        'created_at',
    ]
    available = [c for c in cols if c in friday_alerts.columns]
    print(friday_alerts[available].head(15).to_string(index=False))


## 8-K Monthly Spike Signal Review

Sanity checks:
- Count should exceed baseline mean + 2σ.
- Baseline includes zero-months in the lookback window.


In [ ]:
spike_alerts = load_alerts('8K_SPIKE')
if spike_alerts.empty:
    print('No 8-K spike alerts yet. Run src/detection/8k_spike_detection.py first.')
else:
    cols = [
        'company_ticker',
        'company_name',
        'month',
        'count',
        'baseline_mean',
        'baseline_std',
        'threshold',
        'severity_score',
        'created_at',
    ]
    available = [c for c in cols if c in spike_alerts.columns]
    print(spike_alerts[available].head(20).to_string(index=False))

    print('Top spikes by count:')
    print(spike_alerts.sort_values('count', ascending=False)[available].head(10).to_string(index=False))


## Spot-Check Helper (per company)

Use this to inspect filing volume by month for a specific company.


In [ ]:
def show_company_8k_history(cik: int, months: int = 6) -> pd.DataFrame:
    cutoff_date = (pd.Timestamp.utcnow().normalize() - pd.DateOffset(months=months)).date().isoformat()
    df = read_sql_df(
        """
        SELECT filed_date
        FROM filing_events
        WHERE cik = :cik
          AND filing_type IN ('8-K', '8-K/A')
          AND filed_date >= :cutoff_date
        """,
        {'cik': cik, 'cutoff_date': cutoff_date},
    )

    if df.empty:
        print('No filings found for this CIK in the lookback window.')
        return df

    df['filed_date'] = pd.to_datetime(df['filed_date'], format='mixed', utc=True)
    monthly = (
        df.set_index('filed_date')
          .resample('MS')
          .size()
          .rename('count')
    )
    return monthly.reset_index()

# Example:
# show_company_8k_history(320193)
